## **Problem Statement**

### Business Context

Northbridge Bank is a mid-to-large commercial bank whose core business is providing loans and credit facilities to businesses, generating revenue primarily through interest income and related lending fees while managing the associated credit risk. It has a $6–18 billion lending portfolio covering approximately 2,000–8,000 borrower accounts across 8–12 industry sectors. Its Credit Risk Analytics team supports the CRO, Board Risk Management Committee, Credit Committee, internal audit, and regulatory teams, handling 15–20 ad-hoc portfolio data requests per day in addition to scheduled reporting.
Most requests are routine questions such as sector-wise exposure, overdue accounts above a threshold, or top borrowers by outstanding amount. Although analysts already know the required tables, joins, and business rules, each request takes 2–4 hours because they must manually interpret the question, write and execute SQL, validate results, format the output, and respond. Around 60–70% are repetitive variations of previously answered questions.

This repetitive workload reduces the team's capacity for higher-value activities such as model validation, stress testing, portfolio strategy, and emerging-risk identification. The bank therefore wants business users to obtain routine portfolio answers directly, while ensuring accuracy, transparent logic, auditability, and human escalation for complex or ambiguous questions.

The solution must operate within the bank's existing environment. Of 10 analysts, six are proficient in SQL and only two routinely handle complex joins and credit-risk calculations. Business users do not understand SQL, while analysts must retain control over complex requests. Credit-risk data is stored in a controlled analytical database, and the solution must operate in read-only mode. Customer-identifiable information must not be exposed through unrestricted or public LLM services; approved enterprise-grade LLM APIs may be used subject to the bank’s security and data-governance requirements. All AI-generated outputs must remain reviewable and auditable.

Previous approaches - a shared SQL library, BI dashboards, and Excel templates - addressed parts of the problem but had limitations. The SQL library became unreliable after schema changes, dashboards supported only predefined questions and required BI development for new requests, and Excel templates created dependency on individual analysts. None provided a unified solution for recurring queries, ad-hoc questions, and auditability.


### Objective

Build an internal natural-language query engine that enables business users to ask routine commercial lending portfolio questions in plain English and receive verified answers in minutes, while allowing analysts to focus on complex, judgment-intensive work.

The system will:

- Use pre-approved SQL templates for recurring queries that are tested, version-controlled, and updated when schemas change.
- Generate SQL for uncovered questions, restricted to read-only SELECT queries.
- Validate generated SQL for correctness and retry once if validation fails before escalating to a human analyst.
- Display the SQL query used, raw data returned, and a confidence score with every answer.
- Maintain a complete audit trail for every analytical output.

The initiative is prioritized this quarter due to increasing demand for faster risk reporting, fixed specialist headcount, and regulatory and governance requirements for timely, reproducible, and auditable portfolio-risk answers. Continuing the manual model would increase workload, deepen dependency on specialists, and delay responses during time-sensitive risk or regulatory events.


**Target Outcomes - Within One Quarter**

- Recurring query turnaround: under 2 minutes.
- Ad-hoc query turnaround: under 5 minutes for queries that do not require human review.
- Maintain a complete audit trail for every system-generated analytical output.

**PoC Scope**

The PoC covers only the commercial lending portfolio and read-only analytical queries. It will not perform database updates or write operations and will not replace human judgment for complex credit-risk decisions.


### Data Description



The dataset consists of four database tables supporting commercial lending and credit-risk analysis, along with a test-case CSV used to evaluate the query engine.

- The database supports analysis of a commercial lending portfolio, including sector exposure, delinquency, restructuring, interest-rate analysis, credit ratings, and IFRS 9 provisioning.

- The `test_queries.csv` file provides the ground-truth cases used to evaluate whether the query engine selects the appropriate route and verified query where applicable, and produces the expected analytical output.

#### credit_risk_portfolio.db

**sector_master**: Reference table for industry sector classification

* **sector_code**: Internal sector code (for example, SEC_RE, SEC_INFRA)
* **sector_name**: Human-readable sector name
* **naics_code**: North American Industry Classification System code
* **naics_description**: Description of the NAICS classification
* **is_sensitive_sector**: Flag indicating regulatory-sensitive sectors

**loan_master**: Central loan-level table containing borrower, loan, financial, and delinquency information

* **loan_account_number**: Unique loan identifier
* **borrower_id, borrower_name, borrower_type, group_name, state**: Borrower attributes
* **product_type, loan_category, sector_code**: Loan classification
* **sanctioned_amount, disbursed_amount, outstanding_principal, outstanding_interest, total_outstanding**: Financial amounts
* **interest_rate, rate_type, sanction_date, maturity_date, repayment_frequency**: Loan terms
* **is_consortium, is_restructured, restructuring_date, is_secured**: Loan flags
* **days_past_due, asset_classification, classification_date**: Delinquency and asset-classification information

**borrower_rating**: Credit-rating history maintained for each borrower across assessment dates

* **borrower_id, rating_date**: Borrower and assessment identifiers
* **internal_rating, previous_rating, rating_direction**: Rating information and movement
* **external_rating_agency, external_rating**: External rating information
* **pd_estimate**: Probability of default estimate

**provisioning**: IFRS 9 staging and expected credit loss information by loan and reporting date

* **loan_account_number, reporting_date**: Loan and reporting-period identifiers
* **ifrs9_stage, stage_rationale**: IFRS 9 stage and rationale
* **pd_12_month, pd_lifetime, lgd_estimate, ead_amount, ecl_amount**: Provisioning and credit-risk metrics
* **provision_held, provision_coverage_ratio, is_individually_assessed**: Provision details


**Reporting Timeline & Metadata**

* **Provisioning Dates**: 2024-12-31, 2025-03-31, 2025-06-30, 2025-09-30 *(Latest: 2025-09-30)*
* **Rating Dates**: 2024-09-30, 2024-12-31, 2025-03-31, 2025-06-30, 2025-09-30 *(Latest: 2025-09-30)*
* **NPA Definition**: `asset_classification IN ('Substandard', 'Doubtful', 'Loss')`

#### test_queries.csv




Evaluation dataset containing predefined test cases and their expected outcomes

* **Test Case**: Unique identifier for each evaluation case
* **User Query**: Natural-language question submitted to the query engine
* **Expected Route**: Expected processing path - verified or generated
* **Expected Query ID**: Expected verified-query template where applicable
* **Expected Answer**: Expected result or answer characteristics used for evaluation


## **Please read the instructions carefully before starting the project.**

This is a Python Notebook file with blocks of code pre-filled in alignment with a possible solution for the business use case at hand, and some blank sections where the code has to be written from scratch.

* Please feel free to
    * leverage the pre-filled code blocks as they are, or
    * update the pre-filled code blocks to incorporate necessary changes as per your desired solution workflow for the business problem at hand, or
    * discard the pre-filled code blocks and write the entire code from scratch
* Notebook sections and code blocks that contain instructions and tasks to be performed are mentioned.
* Blanks '\_\_\_\_\_' are provided in the notebook that need to be filled with an appropriate code to get the correct result.
    * With every '\_\_\_\_\_' blank, there is a comment that briefly describes what needs to be filled in the blank space.
* Identify the task to be performed correctly, and only then proceed to write the required code.
* Please sequentially run the code cells from the beginning to avoid any unnecessary errors.
* Add the results/observations derived from the analysis under the respective notebook sections as per the grading rubric requirements.

## **Installing and Importing Necessary Libraries and Dependencies**

In [1]:
!pip install -q langchain langchain-core langchain-openai pandas numpy sqlparse

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 14.7 MB/s eta 0:00:00


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for VSCode/Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [2]:
from typing import TypedDict, List, Any, Dict, Optional  # Type hints and structured data types
import pandas as pd  # Data manipulation and analysis
import sqlite3  # Connect to and query the SQLite database
import sqlparse  # Parse and format SQL queries
import json  # Read and write JSON data
import re  # Perform regular expression-based text processing
import os  # Interact with the operating system and environment variables


from langchain_openai import ChatOpenAI  # Use OpenAI chat models through LangChain
from langchain_core.prompts import ChatPromptTemplate  # Create reusable structured prompts
from langchain_core.messages import HumanMessage, SystemMessage  # Define system and user messages for LLM calls

import warnings  # Manage Python warning messages
warnings.filterwarnings('ignore')  # Suppress warning messages to keep notebook output clean

## **Data Loading and Model Initialization**



### OpenAI API Calling

We load OpenAI credentials from a secure config.json file and store them as environment variables.



In [3]:
# Load the JSON file and extract values
file_name = 'config.json'                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    OPENAI_API_KEY = config.get("OPENAI_API_KEY")                               # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")                             # Extract the OpenAI base URL from the config

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY                                   # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE                                 # Set API base URL as environment variable

In [45]:
llm = ChatOpenAI(model="gpt-4o", temperature=0.0)
evaluator_llm = ChatOpenAI(model="gpt-4o", temperature=0.0)


### Database Loading

In [8]:
import sqlite3

db_path = 'credit_risk_portfolio.db'
conn = sqlite3.connect(f'file:{db_path}?mode=ro', uri=True)

print("Database connection established (read-only mode)")

Database connection established (read-only mode)


**Connect to the SQLite Database**

Create a **read-only connection** to the provided SQLite database using `sqlite3`. Use URI mode with `?mode=ro` to prevent any write operations.

**Sample code:**

```python
import sqlite3

conn = sqlite3.connect(f'file:{db_path}?mode=ro', uri=True)

print("Database connection established (read-only mode)")
```


In [9]:
# preview each table

Define the database schema that will be provided to the LLM in the prompts.

In [10]:
database_schema = """
sector_master:
  sector_code (TEXT, PK): internal sector identifier (e.g., SEC_RE, SEC_INFRA)
  sector_name (TEXT): human-readable sector name (e.g., Real Estate, Infrastructure)
  naics_code (TEXT): NAICS industry classification code
  naics_description (TEXT): NAICS code description
  is_sensitive_sector (INTEGER): 1 if sensitive sector, 0 otherwise

loan_master:
  loan_account_number (TEXT, PK): unique loan identifier
  borrower_id (TEXT): borrower identifier (joins to borrower_rating.borrower_id)
  borrower_name (TEXT): registered legal name of the borrower
  borrower_type (TEXT): entity type (C-Corporation, S-Corporation, LLC, LP, Partnership, Sole Proprietorship)
  group_name (TEXT): business group affiliation, NULL if standalone
  state (TEXT): state of registered office
  product_type (TEXT): Term Loan, Working Capital, Cash Credit, Overdraft, Bill Discounting, Letter of Credit
  loan_category (TEXT): Corporate, Mid-Corporate, SME
  sector_code (TEXT, FK): joins to sector_master.sector_code
  sanctioned_amount (REAL): original approved loan amount in USD
  disbursed_amount (REAL): total amount disbursed in USD
  outstanding_principal (REAL): current principal outstanding in USD
  outstanding_interest (REAL): accrued interest outstanding in USD
  total_outstanding (REAL): outstanding_principal + outstanding_interest in USD
  interest_rate (REAL): current interest rate as percentage
  rate_type (TEXT): Fixed, Floating, MCLR-linked, Repo-linked
  sanction_date (DATE): date of original sanction
  maturity_date (DATE): contractual maturity date
  repayment_frequency (TEXT): Monthly, Quarterly, Bullet
  branch_code (TEXT): originating branch identifier
  branch_name (TEXT): originating branch name
  relationship_manager (TEXT): assigned relationship manager name
  is_consortium (INTEGER): 1 if consortium loan, 0 otherwise
  is_restructured (INTEGER): 1 if restructured, 0 otherwise
  restructuring_date (DATE): date of last restructuring, NULL if not restructured
  is_secured (INTEGER): 1 if secured, 0 if unsecured
  days_past_due (INTEGER): current maximum days past due for the loan
  asset_classification (TEXT): Pass, Special Mention, Substandard, Doubtful, Loss
  classification_date (DATE): date current classification was assigned

borrower_rating:
  rating_id (INTEGER, PK): auto-increment identifier
  borrower_id (TEXT, FK): joins to loan_master.borrower_id
  rating_date (DATE): date of rating assessment
  internal_rating (TEXT): bank's internal rating grade (AAA through D, 18-grade scale)
  previous_rating (TEXT): rating grade from prior assessment
  rating_direction (TEXT): Upgraded, Downgraded, Maintained
  external_rating_agency (TEXT): S&P, Moody's, Fitch, DBRS Morningstar, Kroll, or NULL
  external_rating (TEXT): external agency rating
  pd_estimate (REAL): probability of default (decimal, e.g., 0.02 for 2%)
  rating_model_version (TEXT): internal rating model version

provisioning:
  provision_id (INTEGER, PK): auto-increment identifier
  loan_account_number (TEXT, FK): joins to loan_master.loan_account_number
  reporting_date (DATE): quarter-end reporting date
  ifrs9_stage (INTEGER): IFRS 9 stage (1, 2, or 3)
  stage_rationale (TEXT): reason for stage assignment
  pd_12_month (REAL): 12-month probability of default
  pd_lifetime (REAL): lifetime probability of default
  lgd_estimate (REAL): loss given default (decimal)
  ead_amount (REAL): exposure at default in USD
  ecl_amount (REAL): expected credit loss in USD
  provision_held (REAL): provision amount held in USD
  provision_coverage_ratio (REAL): provision_held / total_outstanding * 100
  is_individually_assessed (INTEGER): 1 if individually assessed, 0 if modeled

Available reporting_date values in provisioning: 2024-12-31, 2025-03-31, 2025-06-30, 2025-09-30
Available rating_date values in borrower_rating: 2024-09-30, 2024-12-31, 2025-03-31, 2025-06-30, 2025-09-30
Latest reporting_date: 2025-09-30
Latest rating_date: 2025-09-30
NPA definition: asset_classification IN ('Substandard', 'Doubtful', 'Loss')
"""

### CSV Loading

In [46]:
ground_truth = pd.read_csv('test_queries.csv')

## **Verified Query Template Library**



The Verified Query Template Library contains ten pre-approved SQL queries covering the most common recurring questions in credit risk analytics. Each template returns a complete result set, and the LLM writes a focused narrative from the full result based on what the user actually asked.

Each entry has a unique identifier, a plain-English description used for intent matching, and a fixed SQL statement that runs without modification.

### Verified Query 1

VQ1: Sector-wise Outstanding and NPA Breakdown

Calculate the total outstanding exposure and NPA exposure for each sector.

**Steps:**

1. Use `loan_master` for loan and outstanding information.
2. Join `sector_master` using `sector_code` to get the sector name.
3. Calculate total `total_outstanding` for each sector.
4. Calculate NPA exposure using `asset_classification` values `Substandard`, `Doubtful`, and `Loss`.
5. Convert both amounts to millions.
6. Group the results by sector.
7. Sort by total outstanding exposure in descending order.


In [15]:
sql_1 = """SELECT
  sm.sector_name,
  ROUND(SUM(lm.total_outstanding) / 1000000.0, 2) AS total_outstanding_million,
  ROUND(SUM(CASE WHEN lm.asset_classification IN ('Substandard', 'Doubtful', 'Loss') THEN lm.total_outstanding ELSE 0 END) / 1000000.0, 2) AS npa_exposure_million
FROM
  loan_master AS lm
JOIN
  sector_master AS sm ON lm.sector_code = sm.sector_code
GROUP BY
  sm.sector_name
ORDER BY
  total_outstanding_million DESC;"""

### Verified Query 2

VQ2: Portfolio Outstanding by Loan Category

Calculate the total outstanding exposure and loan count for each loan category.

**Steps:**

1. Use the `loan_master` table.
2. Group the loans by `loan_category`.
3. Calculate the sum of `total_outstanding` for each category.
4. Count the number of loans in each category.
5. Convert outstanding exposure to millions.
6. Sort the results by outstanding exposure in descending order.


In [17]:
sql_2 = """SELECT
  loan_category,
  ROUND(SUM(total_outstanding) / 1000000.0, 2) AS total_outstanding_million,
  COUNT(loan_account_number) AS loan_count
FROM
  loan_master
GROUP BY
  loan_category
ORDER BY
  total_outstanding_million DESC;"""

### Verified Query 3

VQ3: IFRS 9 Stage-wise ECL Summary

Summarize loan exposure and expected credit loss by IFRS 9 stage for the latest reporting quarter.

**Steps:**

1. Use the `provisioning` table.
2. Filter records for `reporting_date = '2025-09-30'`.
3. Group the results by `ifrs9_stage`.
4. Count the number of loans in each stage.
5. Calculate total `ead_amount` and `ecl_amount` for each stage.
6. Convert EAD and ECL to millions.


In [18]:
sql_3 = """SELECT
  ifrs9_stage,
  COUNT(DISTINCT loan_account_number) AS loan_count,
  ROUND(SUM(ead_amount) / 1000000.0, 2) AS total_ead_million,
  ROUND(SUM(ecl_amount) / 1000000.0, 2) AS total_ecl_million
FROM
  provisioning
WHERE
  reporting_date = '2025-09-30'
GROUP BY
  ifrs9_stage
ORDER BY
  ifrs9_stage;"""

### Verified Query 4

VQ4: Provision Coverage Ratio by Sector

Calculate the average Provision Coverage Ratio for each sector for the latest reporting quarter.

**Steps:**

1. Use `provisioning` for provision coverage information.
2. Join `loan_master` using `loan_account_number`.
3. Join `sector_master` using `sector_code`.
4. Filter for `reporting_date = '2025-09-30'`.
5. Calculate the average `provision_coverage_ratio` for each sector.
6. Group the results by sector.
7. Sort by average coverage ratio in descending order.


In [22]:
sql_4 = """SELECT
  sm.sector_name,
  ROUND(AVG(p.provision_coverage_ratio), 2) AS average_provision_coverage_ratio
FROM
  provisioning AS p
JOIN
  loan_master AS lm ON p.loan_account_number = lm.loan_account_number
JOIN
  sector_master AS sm ON lm.sector_code = sm.sector_code
WHERE
  p.reporting_date = '2025-09-30'
GROUP BY
  sm.sector_name
ORDER BY
  average_provision_coverage_ratio DESC;"""

### Verified Query 5

VQ5: Top 10 Loan Exposures

Identify the 10 individual loans with the highest outstanding exposure.

**Steps:**

1. Use the `loan_master` table.
2. Select the borrower name, sector, outstanding amount, and asset classification.
3. Use `total_outstanding` to represent the loan exposure.
4. Convert outstanding exposure to millions.
5. Sort loans by outstanding exposure in descending order.
6. Return only the top 10 loans.


In [23]:
sql_5 = """SELECT
  lm.borrower_name,
  sm.sector_name,
  ROUND(lm.total_outstanding / 1000000.0, 2) AS total_outstanding_million,
  lm.asset_classification
FROM
  loan_master AS lm
JOIN
  sector_master AS sm ON lm.sector_code = sm.sector_code
ORDER BY
  lm.total_outstanding DESC
LIMIT 10;"""

### Verified Query 6

VQ6: Top 5 Business Group Exposures

Identify the five business groups with the highest total outstanding exposure.

**Steps:**

1. Use the `loan_master` table.
2. Exclude records where `group_name` is missing.
3. Group the loans by `group_name`.
4. Count the loans within each group.
5. Calculate total `total_outstanding` for each group.
6. Convert exposure to millions.
7. Sort groups by total exposure in descending order.
8. Return only the top five groups.


In [24]:
sql_6 = """SELECT
  group_name,
  COUNT(loan_account_number) AS loan_count,
  ROUND(SUM(total_outstanding) / 1000000.0, 2) AS total_outstanding_million
FROM
  loan_master
WHERE
  group_name IS NOT NULL
GROUP BY
  group_name
ORDER BY
  total_outstanding_million DESC
LIMIT 5;"""

### Verified Query 7

VQ7: All Overdue Loan Accounts

Identify all loans that are currently overdue and show their delinquency information.

**Steps:**

1. Use the `loan_master` table.
2. Filter for loans where `days_past_due > 0`.
3. Return the loan account number, borrower name, sector, outstanding amount, DPD, and asset classification.
4. Convert outstanding exposure to millions.
5. Sort the results by `days_past_due` in descending order.


In [25]:
sql_7 = """SELECT
  lm.loan_account_number,
  lm.borrower_name,
  sm.sector_name,
  ROUND(lm.total_outstanding / 1000000.0, 2) AS total_outstanding_million,
  lm.days_past_due,
  lm.asset_classification
FROM
  loan_master AS lm
JOIN
  sector_master AS sm ON lm.sector_code = sm.sector_code
WHERE
  lm.days_past_due > 0
ORDER BY
  lm.days_past_due DESC;"""

### Verified Query 8

VQ8: DPD Bucket Distribution

Show how loans and outstanding exposure are distributed across different Days Past Due buckets.

**Steps:**

1. Use the `loan_master` table.
2. Create the following buckets based on `days_past_due`:

   * `0 (Current)` for 0 days
   * `1-30` for 1 to 30 days
   * `31-60` for 31 to 60 days
   * `61-90` for 61 to 90 days
   * `90+` for more than 90 days
3. Count the loans in each bucket.
4. Calculate total outstanding exposure for each bucket.
5. Convert exposure to millions.
6. Order the buckets from current to highest DPD.


In [27]:
sql_8 = """SELECT
  CASE
    WHEN days_past_due = 0 THEN '0 (Current)'
    WHEN days_past_due BETWEEN 1 AND 30 THEN '1-30'
    WHEN days_past_due BETWEEN 31 AND 60 THEN '31-60'
    WHEN days_past_due BETWEEN 61 AND 90 THEN '61-90'
    ELSE '90+'
  END AS dpd_bucket,
  COUNT(loan_account_number) AS loan_count,
  ROUND(SUM(total_outstanding) / 1000000.0, 2) AS total_outstanding_million
FROM
  loan_master
GROUP BY
  dpd_bucket
ORDER BY
  CASE
    WHEN dpd_bucket = '0 (Current)' THEN 0
    WHEN dpd_bucket = '1-30' THEN 1
    WHEN dpd_bucket = '31-60' THEN 2
    WHEN dpd_bucket = '61-90' THEN 3
    ELSE 4
  END;"""

### Verified Query 9

VQ9: Latest Rating Downgrades

Identify borrowers whose internal credit rating was downgraded in the latest rating cycle.

**Steps:**

1. Use the `borrower_rating` table.
2. Filter for `rating_date = '2025-09-30'`.
3. Filter for `rating_direction = 'Downgraded'`.
4. Return borrower ID, previous rating, current internal rating, and PD estimate.
5. Sort the results by `pd_estimate` in descending order.


In [31]:
sql_9 = """SELECT
  borrower_id,
  previous_rating,
  internal_rating,
  pd_estimate
FROM
  borrower_rating
WHERE
  rating_date = '2025-09-30' AND rating_direction = 'Downgraded'
ORDER BY
  pd_estimate DESC;"""

### Verified Query 10

VQ10: ECL Trend Across Reporting Quarters

Show how total Expected Credit Loss has changed across reporting quarters.

**Steps:**

1. Use the `provisioning` table.
2. Group the records by `reporting_date`.
3. Calculate the total `ecl_amount` for each reporting date.
4. Convert ECL to millions.
5. Sort the results chronologically by reporting date.


In [32]:
sql_10 = """SELECT
  reporting_date,
  ROUND(SUM(ecl_amount) / 1000000.0, 2) AS total_ecl_million
FROM
  provisioning
GROUP BY
  reporting_date
ORDER BY
  reporting_date;"""

### Verified Query Library

The below code creates a dictionary-based **Verified Query Library**, where each query ID (`VQ1`, `VQ2`, etc.) is mapped to its description and corresponding SQL query.


In [33]:
verified_query_library = {
    'VQ1': {
        'description': 'Sector-wise total outstanding and NPA amount breakdown across all sectors',
        'sql': sql_1
    },


    'VQ2': {
        'description': 'Total portfolio outstanding broken down by loan category (Corporate, Mid-Corporate, SME)',
        'sql': sql_2
    },


    'VQ3': {
        'description': 'IFRS 9 stage-wise summary showing loan count, exposure at default, and expected credit loss for the latest quarter',
        'sql': sql_3
    },


    'VQ4': {
        'description': 'Average provision coverage ratio by sector for the latest reporting quarter',
        'sql': sql_4
    },


    'VQ5': {
        'description': 'Top 10 largest loan exposures by outstanding amount at the borrower level',
        'sql': sql_5
    },


    'VQ6': {
        'description': 'Top 5 largest exposures aggregated at the business group level',
        'sql': sql_6
    },


    'VQ7': {
        'description': 'All overdue loan accounts with their days past due and asset classification',
        'sql': sql_7
    },


    'VQ8': {
        'description': 'Distribution of loans across days-past-due buckets showing aging profile of the portfolio',
        'sql': sql_8
    },


    'VQ9': {
        'description': 'Borrowers whose internal rating was downgraded in the latest rating cycle',
        'sql': sql_9
    },


    'VQ10': {
        'description': 'Expected credit loss trend across all reporting quarters showing provisioning movement over time',
        'sql': sql_10
    }
}

print(f"Verified query library loaded with {len(verified_query_library)} templates")
for qid, entry in verified_query_library.items():
    print(f"  {qid}: {entry['description']}")

Verified query library loaded with 10 templates
  VQ1: Sector-wise total outstanding and NPA amount breakdown across all sectors
  VQ2: Total portfolio outstanding broken down by loan category (Corporate, Mid-Corporate, SME)
  VQ3: IFRS 9 stage-wise summary showing loan count, exposure at default, and expected credit loss for the latest quarter
  VQ4: Average provision coverage ratio by sector for the latest reporting quarter
  VQ5: Top 10 largest loan exposures by outstanding amount at the borrower level
  VQ6: Top 5 largest exposures aggregated at the business group level
  VQ7: All overdue loan accounts with their days past due and asset classification
  VQ8: Distribution of loans across days-past-due buckets showing aging profile of the portfolio
  VQ9: Borrowers whose internal rating was downgraded in the latest rating cycle
  VQ10: Expected credit loss trend across all reporting quarters showing provisioning movement over time


## **Tool Definitions**



Each stage of the query engine pipeline is implemented as a modular function. This separation of concerns ensures that:

- Routing decisions are decoupled from query construction
- Query construction is decoupled from validation
- Validation is decoupled from execution
- Every stage can be inspected, tested, and modified independently

### Intent Classification Tool



This tool takes the user’s natural-language query and routes it to either a verified query template or fresh SQL generation based on semantic intent matching.


In [38]:
def classify_intent(user_question, query_library):
    '''
    Classifies the user question and decides which route to take.

    Parameters:
    - user_question (str): The natural language question from the user.
    - query_library (dict): The verified query template library.

    Returns:
    - dict: Contains 'route' (verified or generated),
                     'query_id' (template ID or None),
                     'match_reason' (short explanation of the decision).
    '''

    library_descriptions = '\n'.join(
        [f"{qid}: {entry['description']}" for qid, entry in query_library.items()]
    )

    classification_prompt = f"""
You are an expert in SQL and financial risk analysis. Your task is to classify a user's natural language question into one of two routes: 'verified' or 'generated'.

'verified' route: Choose this if the user's question can be answered by one of the pre-defined SQL queries in the `query_library` below. These are recurring, common questions.
'generated' route: Choose this if the user's question is novel and requires a new SQL query to be generated. This should be a robust, read-only SELECT query.

When classifying, consider the semantic meaning of the user's question and compare it to the descriptions of the verified queries. If there's a clear and direct match, use the 'verified' route. Otherwise, use the 'generated' route.

Think step-by-step. First, analyze the user question. Second, carefully read through the descriptions of all available verified queries. Third, decide if there is a strong semantic match. Fourth, output your decision in the specified JSON format.

User Question:
{user_question}

Available Verified Queries (query_id: description):
{library_descriptions}

### OUTPUT

Return ONLY a valid JSON dictionary with these exact keys:
{{
  "route": "verified" or "generated",
  "query_id": "VQ1" or "VQ2" ... "VQ10" or null,
  "match_reason": "one short sentence explaining the decision"
}}
Do not include any other text.
"""

    response = llm.invoke(classification_prompt).content.strip()
    # Extract JSON from potential markdown blocks
    json_match = re.search(r'\{.*\}', response, re.DOTALL)
    if json_match:
        return json.loads(json_match.group())
    return {"route": "generated", "query_id": None, "match_reason": "Could not parse classification"}

### Query Generation Tool



This tool takes a novel user query and generates a read-only, SQLite-compatible SQL query using the provided database schema.


In [39]:
def generate_query(user_question, schema_context):
    '''
    Generates a candidate SQL query for a novel question using the database schema.

    Parameters:
    - user_question (str): The natural language question.
    - schema_context (str): Full database schema description.

    Returns:
    - str: Candidate SQL query as a string.
    '''

    generation_prompt = f"""
You are an expert in SQL and financial risk analysis. Your task is to write a SQLite SQL query based on a user's question and the provided database schema.

Here are the rules you must follow:
1. The query must be read-only; only use SELECT statements. Do not use any DDL or DML statements.
2. Only use table and column names that exist in the provided schema. Do not invent new ones.
3. Return only the SQL query as a raw string, without any additional text or markdown formatting.
4. Ensure the SQL query directly answers the user's question.
5. When dealing with dates, assume the latest available date for filtering, which is '2025-09-30' for both `reporting_date` in the `provisioning` table and `rating_date` in the `borrower_rating` table, unless the user specifies a different date.
6. For asset classification, 'NPA' refers to `asset_classification IN ('Substandard', 'Doubtful', 'Loss')`.
7. Convert monetary amounts (e.g., total_outstanding, sanctioned_amount, ecl_amount, ead_amount) to millions by dividing by 1000000.0 and rounding to two decimal places, unless otherwise specified by the user.
8. Pay close attention to aggregations and groupings based on the user's request.
9. If the user asks for a 'trend' or 'movement over time', ensure the query includes `reporting_date` or `rating_date` in the SELECT clause and groups by it, ordering chronologically.

Database Schema:
{schema_context}

User Question:
{user_question}

SQL Query:
"""

    sql = llm.invoke(generation_prompt).content.strip()
    # Strip markdown fences if present
    sql = re.sub(r'^```sql\s*|\s*```$', '', sql, flags=re.IGNORECASE | re.MULTILINE).strip()
    sql = re.sub(r'^```\s*|\s*```$', '', sql, flags=re.MULTILINE).strip()
    return sql

### Query Validation Tool


The `validate_query` tool validates the candidate SQL through five checks before execution:

* **Read-only check** - Ensures the query uses only `SELECT`/`WITH` and contains no destructive operations or multiple statements.
* **Schema conformance check** - Verifies that all referenced tables and columns exist in the database schema.
* **Parse & plan check** - Uses SQLite `EXPLAIN` to confirm that the query can be parsed and planned successfully.
* **LLM relevance check** - Evaluates whether the SQL correctly answers the user’s question, including the appropriate tables, metrics, aggregations, and NPA/date logic where applicable.
* **Template integrity check** - For verified queries, ensures that the candidate query maintains the expected output structure of the approved template.

The query is approved for execution only when all applicable validation checks pass.


In [61]:
def validate_query(user_question, candidate_sql, db_connection, query_library, query_id=None):
    '''
    Validates a candidate SQL query through five checks before execution.

    Parameters:
    - user_question (str): The original user question.
    - candidate_sql (str): The SQL query to validate.
    - db_connection: SQLite connection object.
    - query_library (dict): Verified query library (for integrity check).
    - query_id (str, optional): Template ID if from verified track.

    Returns:
    - dict: Contains 'passed' (bool), 'failed_check' (str or None), 'details' (str),
            and 'relevance_confidence' (int, 0-1).
    '''

    result = {
        'passed': False,
        'failed_check': None,
        'details': '',
        'relevance_confidence': None
    }

    # Helper to clean SQL for schema inspection
    def clean_sql_for_schema_inspection(sql):
        # Use sqlparse to get the first statement, which is more robust
        parsed_statements = sqlparse.parse(sql)
        if parsed_statements:
            cleaned_sql = str(parsed_statements[0]).strip().rstrip(';')
        else:
            cleaned_sql = sql.strip().rstrip(';')

        # Remove ORDER BY clause (must be before LIMIT if both exist)
        cleaned_sql = re.sub(r'\s+ORDER BY\s+.*?$', '', cleaned_sql, flags=re.IGNORECASE | re.DOTALL)
        # Remove LIMIT clause
        cleaned_sql = re.sub(r'\s+LIMIT\s+.*?$', '', cleaned_sql, flags=re.IGNORECASE | re.DOTALL)
        return cleaned_sql

    # Check 1: Read-only shape check
    sql_upper = candidate_sql.upper().strip()
    forbidden_keywords = ['DROP', 'DELETE', 'UPDATE', 'INSERT', 'ALTER', 'TRUNCATE', 'REPLACE', 'ATTACH']
    if not (sql_upper.startswith('SELECT') or sql_upper.startswith('WITH')):
        result['failed_check'] = 'read_only_shape'
        result['details'] = 'Query must start with SELECT or WITH'
        return result
    for kw in forbidden_keywords:
        if re.search(r'\b' + kw + r'\b', sql_upper):
            result['failed_check'] = 'read_only_shape'
            result['details'] = f'Forbidden keyword detected: {kw}'
            return result
    if ';' in candidate_sql.rstrip(';').rstrip():
        result['failed_check'] = 'read_only_shape'
        result['details'] = 'Multiple statements are not allowed'
        return result



    # Check 2: Schema conformance check
    cur = db_connection.cursor()
    real_tables = [r[0] for r in cur.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
    real_columns = set()
    for t in real_tables:
        for col_info in cur.execute(f"PRAGMA table_info({t})").fetchall():
            real_columns.add(col_info[1].lower())
    parsed = sqlparse.parse(candidate_sql)[0]
    tokens = [str(t).strip().lower() for t in parsed.flatten() if t.ttype is None or 'Name' in str(t.ttype)]
    referenced_identifiers = re.findall(r'\b[a-z_][a-z0-9_]*\b', candidate_sql.lower())
    sql_keywords = {'select','from','where','and','or','group','by','order','having','limit','join','on','as','case',
                    'when','then','else','end','sum','count','avg','min','max','round','desc','asc','left','right',
                    'inner','outer','distinct','null','is','not','in','like','with','union','all','between','coalesce'}
    unknown = [tok for tok in referenced_identifiers
               if tok not in sql_keywords and tok not in real_columns and tok not in real_tables
               and not tok.isdigit() and tok not in ('s', 'l', 'p', 'r', 'e6')]



    # Check 3: Parse-and-plan dry run using EXPLAIN
    try:
        # Pass the candidate_sql directly to EXPLAIN after stripping trailing semicolon
        sql_for_explain = candidate_sql.strip().rstrip(';')
        cur.execute(f"EXPLAIN {sql_for_explain}") # Removed WHERE 1=0 from here
        cur.fetchall()
    except sqlite3.Error as e:
        result['failed_check'] = 'parse_plan_dry_run'
        result['details'] = f'SQL failed to parse or plan: {str(e)}'
        return result



    # Check 4: LLM relevance check
    is_verified_track = query_id is not None and query_id in query_library
    track_context = (
        "This SQL is a pre-approved VERIFIED TEMPLATE. It is intentionally broad "
        "(e.g., it may return all sectors/categories/stages rather than filtering to "
        "just what the user asked). A separate response-generation step will filter and "
        "highlight the relevant rows afterward. Do NOT fail this query for lacking a "
        "WHERE clause that narrows to the user's specific sector/category/stage — judge "
        "only whether the underlying metric, tables, and aggregation logic match the "
        "question's intent."
        if is_verified_track else
        "This SQL was freshly generated for this specific question and should be "
        "appropriately scoped/filtered to answer it directly."
    )


    relevance_prompt = f"""
You are an expert in SQL and financial risk analysis. Your task is to evaluate whether a given SQL query accurately answers a user's question, considering the provided context.

Here are the rules you must follow:
1. Focus only on the semantic relevance of the SQL to the user's question. Do not check for SQL syntax errors or database schema conformance, as those are handled by other checks.
2. Consider the `track_context` carefully to understand if the query is a verified template (which might be broad) or a freshly generated query (which should be specific).
3. If it's a verified template, assess if the core metric, tables, and aggregation align with the user's intent, even if the filtering isn't exact (as filtering happens later).
4. If it's a freshly generated query, assess if it is appropriately scoped and filtered to directly answer the user's question.
5. Assign a `confidence` score from 0.0 to 1.0, where 1.0 means perfect relevance and 0.0 means no relevance.
6. Provide a short, concise `reason` for your verdict and confidence score.

Track Context:
{track_context}

User Question:
{user_question}

SQL Query:
{candidate_sql}

### OUTPUT
Return ONLY a JSON dictionary:
{{
  "verdict": "yes" or "no",
  "confidence": 0.0 to 1.0,
  "reason": "one short sentence"
}}

"""
    relevance_response = evaluator_llm.invoke(relevance_prompt).content.strip()
    json_match = re.search(r'\{.*\}', relevance_response, re.DOTALL)
    if json_match:
        relevance_json = json.loads(json_match.group())
        result['relevance_confidence'] = relevance_json.get('confidence', 0.0)
        if relevance_json.get('verdict') == 'no' or relevance_json.get('confidence', 0.0) < 0.6:
            result['failed_check'] = 'llm_relevance'
            result['details'] = f"Relevance check failed: {relevance_json.get('reason', 'unknown')}"
            return result



    # Check 5: Verified template integrity check (verified track only)
    if query_id and query_id in query_library:
        expected_sql = query_library[query_id]['sql']

        try:
            # Get description for expected SQL after cleaning and adding LIMIT 0
            cleaned_expected_sql = clean_sql_for_schema_inspection(expected_sql)
            cur.execute(f"{cleaned_expected_sql} LIMIT 0")
            expected_cols = [d[0] for d in cur.description]

            # Get description for candidate SQL after cleaning and adding LIMIT 0
            cleaned_candidate_sql = clean_sql_for_schema_inspection(candidate_sql)
            cur.execute(f"{cleaned_candidate_sql} LIMIT 0")
            actual_cols = [d[0] for d in cur.description]

            if len(expected_cols) != len(actual_cols):
                result['failed_check'] = 'template_integrity'
                result['details'] = f'Expected {len(expected_cols)} columns, got {len(actual_cols)} for {query_id}'
                return result
            # Also check column names for better integrity (case-insensitive and order-independent)
            if set(c.lower() for c in expected_cols) != set(c.lower() for c in actual_cols):
                result['failed_check'] = 'template_integrity'
                result['details'] = f'Column names mismatch for {query_id}: Expected {expected_cols}, got {actual_cols}'
                return result

        except sqlite3.Error as e:
            result['failed_check'] = 'template_integrity'
            result['details'] = f'Template integrity check failed for {query_id}: {str(e)}'
            return result

    result['passed'] = True
    result['details'] = 'All validation checks passed'
    return result

### Retry Generation Tool



This tool takes the failed SQL and validation error, then regenerates a corrected, read-only SQLite query while preserving the original user intent.


In [62]:
def retry_generation(user_question, failed_sql, error_message, schema_context):
    '''
    Regenerates SQL after a validation failure, feeding the error back to the LLM.

    Parameters:
    - user_question (str): The original user question.
    - failed_sql (str): The SQL that failed validation.
    - error_message (str): The specific failure reason.
    - schema_context (str): Database schema description.

    Returns:
    - str: Revised SQL as a string.
    '''

    retry_prompt = f"""
You are an expert in SQL and financial risk analysis. Your previous attempt to generate a SQL query failed validation. Your task is to revise the SQL query based on the original user question, the failed SQL, and the specific error message from the validation.

Here are the rules you must follow:
1. The query must be read-only; only use SELECT statements. Do not use any DDL or DML statements.
2. Only use table and column names that exist in the provided schema. Do not invent new ones.
3. Return only the SQL query as a raw string, without any additional text or markdown formatting.
4. Ensure the revised SQL query directly answers the user's question and addresses the identified error.
5. When dealing with dates, assume the latest available date for filtering, which is '2025-09-30' for both `reporting_date` in the `provisioning` table and `rating_date` in the `borrower_rating` table, unless the user specifies a different date.
6. For asset classification, 'NPA' refers to `asset_classification IN ('Substandard', 'Doubtful', 'Loss')`.
7. Convert monetary amounts (e.g., total_outstanding, sanctioned_amount, ecl_amount, ead_amount) to millions by dividing by 1000000.0 and rounding to two decimal places, unless otherwise specified by the user.
8. Pay close attention to aggregations and groupings based on the user's request.
9. If the user asks for a 'trend' or 'movement over time', ensure the query includes `reporting_date` or `rating_date` in the SELECT clause and groups by it, ordering chronologically.

User Question:
{user_question}

Failed SQL:
{failed_sql}

Validation Error:
{error_message}

Database Schema:
{schema_context}

Revised SQL Query:
"""

    revised_sql = llm.invoke(retry_prompt).content.strip()
    revised_sql = re.sub(r'^```sql\s*|\s*```$', '', revised_sql, flags=re.IGNORECASE | re.MULTILINE).strip()
    revised_sql = re.sub(r'^```\s*|\s*```$', '', revised_sql, flags=re.MULTILINE).strip()
    return revised_sql

### Query Execution Tool



This tool runs only gate-passed queries against the read-only database connection and returns the result as a Pandas DataFrame along with a reasonableness check on the result. This tool does not require an LLM prompt because it is purely deterministic.

In [63]:
def execute_query(validated_sql, db_connection):
    '''
    Executes a gate-passed SQL query and returns the result as a DataFrame.

    Parameters:
    - validated_sql (str): SQL query that has passed all validation checks.
    - db_connection: Read-only SQLite connection object.

    Returns:
    - dict: Contains 'dataframe' (pandas DataFrame), 'reasonable' (bool),
            and 'warnings' (list of warning strings).
    '''

    result = {
        'dataframe': None,
        'reasonable': True,
        'warnings': []
    }

    df = pd.read_sql_query(validated_sql, db_connection)
    result['dataframe'] = df

    # Reasonableness checks
    if df.empty:
        result['warnings'].append('Query returned an empty result')

    for col in df.select_dtypes(include='number').columns:
        if (df[col] < 0).any() and 'deviation' not in col.lower() and 'change' not in col.lower():
            result['warnings'].append(f'Column {col} contains negative values')
        if df[col].isnull().any():
            null_count = df[col].isnull().sum()
            if null_count > len(df) * 0.5:
                result['warnings'].append(f'Column {col} has {null_count} null values')

    if len(result['warnings']) > 2:
        result['reasonable'] = False

    return result

### Response Generation Tool


This tool takes the user’s question and query results, then generates a concise, business-focused response highlighting only the relevant insights and exact figures.


In [64]:
def generate_response(user_question, dataframe, route, query_id=None):
    '''
    Generates a focused natural language response from the query result.

    Parameters:
    - user_question (str): The original user question.
    - dataframe (pd.DataFrame): The full query result.
    - route (str): 'verified' or 'generated'.
    - query_id (str, optional): Template ID if from verified track.

    Returns:
    - str: Natural language response focused on what the user asked.
    '''

    response_prompt = f"""
You are an expert in SQL and financial risk analysis. Your task is to generate a concise, business-focused natural language response to the user's question based on the provided query results. Highlight only the relevant insights and exact figures.

Here are the rules you must follow:
1. If the dataframe is empty, state that no results were found for the query.
2. If the query route was 'verified' and `query_id` is provided, remember that the verified templates are intentionally broad. Your response must filter and highlight ONLY the information directly relevant to the `user_question` from the potentially broader `dataframe`.
3. If the query route was 'generated', the dataframe should already be specific to the `user_question`, so summarize the findings directly.
4. Always include specific numerical values from the dataframe where appropriate to support your insights. Convert all currency values to millions with 2 decimal places, for example: $123,456,789.01 becomes $123.46 million.
5. If a trend is requested, describe the direction and magnitude of the change over time, referencing specific periods and values from the data.
6. Maintain a professional and informative tone.

User Question:
{user_question}

Query Results:
{dataframe.to_string()}

Response:
"""

    narrative = llm.invoke(response_prompt).content.strip()
    return narrative

## **LLM Binding and Pipeline Orchestration**



The individual tools are bound together into a complete pipeline function that chains all stages in the correct order. This function serves as the single entry point for any user question and coordinates the flow between intent classification, query construction, validation, execution, visualization, and response generation.

In [65]:
def run_pipeline(user_question, db_connection, query_library, schema_context, verbose=True):
    '''
    Runs the complete query engine pipeline for a single user question.

    Parameters:
    - user_question (str): The natural language question.
    - db_connection: SQLite connection object.
    - query_library (dict): Verified query template library.
    - schema_context (str): Database schema description.
    - verbose (bool): If True, prints intermediate pipeline stages.

    Returns:
    - dict: Complete pipeline output including narrative, SQL, data, and log.
    '''

    log = {
        'user_question': user_question,
        'route': None,
        'query_id': None,
        'match_reason': None,
        'candidate_sql': None,
        'gate_result': None,
        'retry_used': False,
        'escalated': False,
        'executed_sql': None,
        'row_count': None,
        'confidence': None,
        'narrative': None
    }

    # Step 1: Intent classification
    classification = classify_intent(user_question, query_library)
    log['route'] = classification['route']
    log['query_id'] = classification.get('query_id')
    log['match_reason'] = classification.get('match_reason')

    if verbose:
        print(f"[1] Intent Classification: route={log['route']}, query_id={log['query_id']}")
        print(f"    Reason: {log['match_reason']}")


    # Step 2: Query construction
    if log['route'] == 'verified' and log['query_id'] in query_library:
        candidate_sql = query_library[log['query_id']]['sql']
    else:
        candidate_sql = generate_query(user_question, schema_context)
    log['candidate_sql'] = candidate_sql

    if verbose:
        print(f"[2] Query Construction: {'loaded from library' if log['route']=='verified' else 'generated fresh SQL'}")


    # Step 3: Validation gate
    gate = validate_query(user_question, candidate_sql, db_connection, query_library, log['query_id'])
    log['gate_result'] = gate

    if verbose:
        print(f"[3] Validation Gate: passed={gate['passed']}, relevance_confidence={gate.get('relevance_confidence')}")
        if not gate['passed']:
            print(f"    Failed check: {gate.get('failed_check')}")
            print(f"    Details: {gate.get('details')}")


    # Step 4: Retry once on generated track if validation fails
    if not gate['passed'] and log['route'] == 'generated':
        if verbose:
            print(f"    Retrying: {gate['details']}")
        candidate_sql = retry_generation(user_question, candidate_sql, gate['details'], schema_context)
        log['candidate_sql'] = candidate_sql
        log['retry_used'] = True
        gate = validate_query(user_question, candidate_sql, db_connection, query_library, None)
        log['gate_result'] = gate

        if verbose:
            print(f"    Retry Validation Gate: passed={gate['passed']}, relevance_confidence={gate.get('relevance_confidence')}")
            if not gate['passed']:
                print(f"    Retry failed check: {gate.get('failed_check')}")
                print(f"    Retry details: {gate.get('details')}")


    # Step 5: Escalate if still failing
    if not gate['passed']:
        log['escalated'] = True
        log['narrative'] = f"Query could not be reliably resolved. Escalated to human analyst. Failure: {gate['details']}"
        log['confidence'] = 'ESCALATED'
        if verbose:
            print(f"[!] Escalated to human: {gate['details']}")
        return {'log': log, 'dataframe': None, **log}


    # Step 6: Execute
    log['executed_sql'] = candidate_sql
    exec_result = execute_query(candidate_sql, db_connection)
    df = exec_result['dataframe']
    log['row_count'] = len(df)

    if verbose:
        print(f"[4] Execute: {len(df)} rows returned")
        if exec_result['warnings']:
            print(f"    Warnings: {exec_result['warnings']}")


    # Step 7: Response generation
    narrative = generate_response(user_question, df, log['route'], log['query_id'])
    log['narrative'] = narrative


    # Confidence: carried directly from the validation gate's relevance check (0-1)
    log['confidence'] = gate.get('relevance_confidence')

    if verbose:
        print(f"[6] Response Generation: confidence={log['confidence']}")

    return {'log': log, 'dataframe': df, **log}

Verify the pipeline is wired end-to-end with a quick sanity check on a simple question.

In [66]:
sanity_check = run_pipeline('Show me the top 5 largest loan exposures', conn, verified_query_library, database_schema, verbose=True )   # Write the sample query to test the pipeline
print("\nPipeline sanity check complete.")

[1] Intent Classification: route=verified, query_id=VQ6
    Reason: The question matches VQ6, which provides the top 5 largest exposures at the business group level.
[2] Query Construction: loaded from library
[3] Validation Gate: passed=True, relevance_confidence=1.0
[4] Execute: 5 rows returned
[6] Response Generation: confidence=1.0

Pipeline sanity check complete.


## **Test Cases**



We now execute five test cases to evaluate the query engine end-to-end.
- Three test cases exercise the verified template track and two exercise the generated track.
- Each test case is followed by ground truth values and observations to compare against.

### Test Case 1: Sector-Specific NPA Question


In [59]:
test_1 = run_pipeline(ground_truth['User Query'].iloc[0], conn, verified_query_library, database_schema)

[1] Intent Classification: route=verified, query_id=VQ1
    Reason: The question asks for sector-wise breakdown of real estate and non-performing assets, which matches VQ1.
[2] Query Construction: loaded from library
[3] Validation Gate: passed=True, relevance_confidence=1.0
[4] Execute: 10 rows returned
[6] Response Generation: confidence=1.0


**Response:**

In [60]:
print(f"Confidence: {test_1['confidence']}")
print(f"\nNarrative:\n{test_1['narrative']}")
print(f"\nExecuted SQL:\n{test_1['executed_sql']}")
print(f"\nResult data:")
display(test_1['dataframe'])

Confidence: 1.0

Narrative:
The total outstanding amount in the real estate sector is $1,195.12 million. Of this, $89.93 million is classified as non-performing.

Executed SQL:
SELECT
  sm.sector_name,
  ROUND(SUM(lm.total_outstanding) / 1000000.0, 2) AS total_outstanding_million,
  ROUND(SUM(CASE WHEN lm.asset_classification IN ('Substandard', 'Doubtful', 'Loss') THEN lm.total_outstanding ELSE 0 END) / 1000000.0, 2) AS npa_exposure_million
FROM
  loan_master AS lm
JOIN
  sector_master AS sm ON lm.sector_code = sm.sector_code
GROUP BY
  sm.sector_name
ORDER BY
  total_outstanding_million DESC;

Result data:


,sector_name,total_outstanding_million,npa_exposure_million
0,Manufacturing,1900.36,216.21
1,IT & Services,1622.83,75.14
2,Hospitality,1571.12,163.07
3,Infrastructure,1529.95,92.78
4,Power & Energy,1471.72,178.50
5,Textiles,1369.42,185.82
6,Pharmaceuticals,1260.95,22.43
7,Chemicals,1204.83,55.51
8,Real Estate,1195.12,89.93
9,Automobile,1119.73,122.95


**Observation:**

- The Real Estate sector has a total outstanding amount of 1,195.12 million dollars.
- Of the 1,195.12 million, $89.93 million is classified as non-performing (NPA).
- This implies that while Real Estate is a significant part of the portfolio, it also contributes a notable portion to the NPA.
 -The Real Estate sector has a middling asset quality.

### Test Case 2: DPD Aging Profile

In [69]:
test_2 = run_pipeline(ground_truth['User Query'].iloc[1], conn, verified_query_library, database_schema)

[1] Intent Classification: route=verified, query_id=VQ8
    Reason: The question asks for distribution across DPD buckets, which matches the description of VQ8.
[2] Query Construction: loaded from library
[3] Validation Gate: passed=True, relevance_confidence=1.0
[4] Execute: 5 rows returned
[6] Response Generation: confidence=1.0


**Response:**

In [70]:
print(f"Confidence: {test_2['confidence']}")
print(f"\nNarrative:\n{test_2['narrative']}")
print(f"\nExecuted SQL:\n{test_2['executed_sql']}")
print(f"\nResult data:")
display(test_2['dataframe'])

Confidence: 1.0

Narrative:
The distribution of our overdue book across DPD (Days Past Due) buckets is as follows: 

- The "0 (Current)" bucket has 540 loans with a total outstanding amount of $10,795.43 million.
- The "1-30" DPD bucket includes 53 loans, totaling $909.15 million.
- The "31-60" DPD bucket contains 37 loans with an outstanding amount of $835.88 million.
- The "61-90" DPD bucket has 21 loans, amounting to $522.39 million.
- The "90+" DPD bucket comprises 49 loans with a total outstanding of $1,183.17 million.

This distribution highlights that the majority of the outstanding amount is in the "0 (Current)" bucket, while the "90+" bucket, although having fewer loans, still represents a significant overdue amount.

Executed SQL:
SELECT
  CASE
    WHEN days_past_due = 0 THEN '0 (Current)'
    WHEN days_past_due BETWEEN 1 AND 30 THEN '1-30'
    WHEN days_past_due BETWEEN 31 AND 60 THEN '31-60'
    WHEN days_past_due BETWEEN 61 AND 90 THEN '61-90'
    ELSE '90+'
  END AS dpd_b

,dpd_bucket,loan_count,total_outstanding_million
0,0 (Current),540,10795.43
1,1-30,53,909.15
2,31-60,37,835.88
3,61-90,21,522.39
4,90+,49,1183.17


**Observation:**

- Most loans are current (0 DPD), forming the largest part of the portfolio.
- A significant portion of the overdue book is concentrated in the '90+' DPD bucket ($1,183.17 million across 49 loans), indicating higher risk.
- Loans progressively move into higher delinquency stages.
- The 1-30 DPD has the second highest loans with 53 loans that totals 909.15 million dollars.

### Test Case 3: Restructured Portfolio Analysis

In [73]:
test_3 = run_pipeline(ground_truth['User Query'].iloc[2], conn, verified_query_library, database_schema)

[1] Intent Classification: route=generated, query_id=None
    Reason: The question about restructured loans and impairment percentage does not match any verified query.
[2] Query Construction: generated fresh SQL
[3] Validation Gate: passed=False, relevance_confidence=0.7
    Failed check: llm_relevance
    Details: Relevance check failed: The query incorrectly calculates the impaired percentage per loan instead of overall.
    Retrying: Relevance check failed: The query incorrectly calculates the impaired percentage per loan instead of overall.
    Retry Validation Gate: passed=False, relevance_confidence=0.7
    Retry failed check: llm_relevance
    Retry details: Relevance check failed: The query calculates impaired percentage per loan instead of overall, and lacks a total aggregation.
[!] Escalated to human: Relevance check failed: The query calculates impaired percentage per loan instead of overall, and lacks a total aggregation.


**Response:**

In [74]:
print(f"Confidence: {test_3['confidence']}")
print(f"\nNarrative:\n{test_3['narrative']}")
print(f"\nExecuted SQL:\n{test_3['executed_sql']}")
print(f"\nResult data:")
display(test_3['dataframe'])

Confidence: ESCALATED

Narrative:
Query could not be reliably resolved. Escalated to human analyst. Failure: Relevance check failed: The query calculates impaired percentage per loan instead of overall, and lacks a total aggregation.

Executed SQL:
None

Result data:


None

**Observation:**

-It escalated it to a human analysis because the query couldn't be solved reliably.

-The relevance check failed twice. It couldn't calculate impaired percentage of the overall loan.

### Test Case 4: Average Interest Rate by Sector

In [75]:
test_4 = run_pipeline(ground_truth['User Query'].iloc[3], conn, verified_query_library, database_schema)


[1] Intent Classification: route=generated, query_id=None
    Reason: The question about average interest rate by sector does not match any of the verified queries.
[2] Query Construction: generated fresh SQL
[3] Validation Gate: passed=True, relevance_confidence=1.0
[4] Execute: 10 rows returned
[6] Response Generation: confidence=1.0


**Response:**

In [76]:
print(f"Confidence: {test_4['confidence']}")
print(f"\nNarrative:\n{test_4['narrative']}")
print(f"\nExecuted SQL:\n{test_4['executed_sql']}")
print(f"\nResult data:")
display(test_4['dataframe'])

Confidence: 1.0

Narrative:
The average interest rates by sector, sorted from highest to lowest, are as follows: Pharmaceuticals leads with an average interest rate of 11.16%, followed by Textiles at 10.95%, and IT & Services at 10.87%. Infrastructure and Hospitality sectors have rates of 10.80% and 10.79%, respectively. Chemicals, Power & Energy, and Automobile sectors have rates ranging from 10.75% to 10.62%. Manufacturing and Real Estate sectors have the lowest average interest rates at 10.58% and 10.54%, respectively.

Executed SQL:
SELECT 
    sm.sector_name, 
    ROUND(AVG(lm.interest_rate), 2) AS average_interest_rate
FROM 
    loan_master lm
JOIN 
    sector_master sm ON lm.sector_code = sm.sector_code
GROUP BY 
    sm.sector_name
ORDER BY 
    average_interest_rate DESC;

Result data:


,sector_name,average_interest_rate
0,Pharmaceuticals,11.16
1,Textiles,10.95
2,IT & Services,10.87
3,Infrastructure,10.80
4,Hospitality,10.79
5,Chemicals,10.75
6,Power & Energy,10.73
7,Automobile,10.62
8,Manufacturing,10.58
9,Real Estate,10.54


**Observation:**

- The average interest rate of pharmaceuticals is the highest.

- Manufacturing and real estate have the lowest interest rate with manufactoring being 10.58% and real estate being 10.54%.

- The interest rate doesn't vary much between different sectors.

### Test Case 5: Stage 3 Trend Over Quarters

In [77]:
test_5 = run_pipeline(ground_truth['User Query'].iloc[4], conn, verified_query_library, database_schema)

[1] Intent Classification: route=generated, query_id=None
    Reason: No verified query specifically addresses Stage 3 book size and expected credit loss over multiple quarters.
[2] Query Construction: generated fresh SQL
[3] Validation Gate: passed=True, relevance_confidence=1.0
[4] Execute: 4 rows returned
[6] Response Generation: confidence=1.0


**Response:**

In [78]:
print(f"Confidence: {test_5['confidence']}")
print(f"\nNarrative:\n{test_5['narrative']}")
print(f"\nExecuted SQL:\n{test_5['executed_sql']}")
print(f"\nResult data:")
display(test_5['dataframe'])

Confidence: 1.0

Narrative:
Over the last four quarters, the Stage 3 book size has shown a significant upward trend. Starting from $346.25 million on December 31, 2024, it increased to $945.87 million by March 31, 2025. This growth continued, reaching $1,175.23 million on June 30, 2025, and further rising to $1,208.23 million by September 30, 2025.

Similarly, the expected credit loss (ECL) associated with the Stage 3 book has also increased. It was $131.73 million at the end of December 2024, then rose sharply to $429.46 million by the end of March 2025. The ECL continued to grow, reaching $502.85 million by June 30, 2025, and $512.40 million by September 30, 2025. This indicates a consistent increase in both the Stage 3 book size and its associated credit risk over the observed period.

Executed SQL:
SELECT 
    p.reporting_date, 
    ROUND(SUM(CASE WHEN p.ifrs9_stage = 3 THEN p.ead_amount ELSE 0 END) / 1000000.0, 2) AS stage_3_book_size_millions,
    ROUND(SUM(CASE WHEN p.ifrs9_stag

,reporting_date,stage_3_book_size_millions,stage_3_ecl_millions
0,2024-12-31,346.25,131.73
1,2025-03-31,945.87,429.46
2,2025-06-30,1175.23,502.85
3,2025-09-30,1208.23,512.40


**Observation:**

- The stage 3 book size has almost quadrupled over the years from 346.25 to 1208.23.
- The growth is mostly frontloaded. The majority of the growth occurs in December 2024 to March 2025.

## **Evaluation Against Ground Truth**


Each test case output is compared against the corresponding ground-truth values to verify whether the correct route and query template were selected and whether the numeric answer aligns with expectations. This evaluation:

* Calculates **Selected Path Accuracy**, **Selected Query Accuracy**, and **Average Confidence Score**.
* Summarizes each test case with its route, query ID, confidence score, and rows returned.


In [81]:
evaluation_rows = []

test_results = [test_1, test_2, test_3, test_4, test_5]

for i, (_, gt) in enumerate(ground_truth.iterrows()):
    tr = test_results[i]

    evaluation_rows.append({
        'Test Case': gt['Test Case'],
        'Expected Route': gt['Expected Route'],
        'Actual Route': tr['route'],
        'Route Match': tr['route'] == gt['Expected Route'],
        'Expected Query ID': gt['Expected Query ID'],
        'Actual Query ID': tr['query_id'],
        'Query ID Match': (
            pd.isna(gt['Expected Query ID']) and pd.isna(tr['query_id'])
        ) or tr['query_id'] == gt['Expected Query ID'],
        'Confidence': tr['confidence'],
        'Rows Returned': tr['row_count']
    })

evaluation_df = pd.DataFrame(evaluation_rows)

path_accuracy = evaluation_df['Route Match'].mean() * 100

verified = evaluation_df['Expected Route'].str.strip().str.lower() == 'verified'
query_accuracy = evaluation_df.loc[verified, 'Query ID Match'].mean() * 100

# Fix: coerce non-numeric confidence values (e.g. "ESCALATED") to NaN so they're
# excluded from the average rather than crashing or corrupting the mean
numeric_confidence = pd.to_numeric(evaluation_df['Confidence'], errors='coerce')
average_confidence = numeric_confidence.mean()
escalated_count = evaluation_df['Confidence'].astype(str).eq('ESCALATED').sum()

print(f"Selected Path Accuracy: {path_accuracy:.1f}%")
print(f"Selected Query Accuracy: {query_accuracy:.1f}%")
print(f"Average Confidence Score: {average_confidence:.2f} (excludes {escalated_count} escalated case(s))")

display(evaluation_df)

Selected Path Accuracy: 100.0%
Selected Query Accuracy: 100.0%
Average Confidence Score: 1.00 (excludes 1 escalated case(s))


,Test Case,Expected Route,Actual Route,Route Match,Expected Query ID,Actual Query ID,Query ID Match,Confidence,Rows Returned
0,TC-01,verified,verified,True,VQ1,VQ1,True,1.0,10.0
1,TC-02,verified,verified,True,VQ8,VQ8,True,1.0,5.0
2,TC-03,generated,generated,True,NaN,None,True,ESCALATED,NaN
3,TC-04,generated,generated,True,NaN,None,True,1.0,10.0
4,TC-05,generated,generated,True,NaN,None,True,1.0,4.0


**Observation:**

- The 5 test cases were routed correctly.

- The average confidence score was 1.0 but TC-03 confidence was escalated therefore, the confidence average might not be as reliable.

- The system decided to not answer instead of giving a bad number to the customer.

- Query ID Match is only a meaningful test in the 2 cases where a template was actually expected (TC-01 and TC-02, both correctly matched to VQ1 and VQ8). The other 3 generated-route cases it it's True is trivial becuase they didn't need to match a template, so the true template-selection accuracy is 100% on 2/2 applicable cases.

## **Deployment**

**Deploy the Application Using GitHub and Streamlit**

In this section, you will deploy your application using **GitHub** and **Streamlit**. A step-by-step deployment guide is provided to help you complete the process.

To generate the `app.py` file, you can use the **free version of Claude**. Download this notebook and upload it to Claude along with the prompt below. Claude will use the notebook content to generate the Streamlit application code.

**Sample Prompt to use in Claude:**

> I have uploaded my Jupyter Notebook containing the complete implementation of my application. Please analyze the notebook and create a complete `app.py` file that converts this notebook-based application into a Streamlit app.
>
> Preserve the existing logic, SQL queries, verified query library, and application workflow. Make the necessary changes to adapt the notebook code for Streamlit, including appropriate user inputs, outputs, and UI components.
>
> Return the complete, ready-to-run `app.py` code in a single code block. Do not omit or replace any important implementation details. Also mention any additional files, packages, environment variables, or configuration required to run the application.

**Deployment steps:**

1. Download this notebook to your computer.
2. Upload the downloaded notebook to the **free version of Claude** along with the prompt above.
3. Review and download the generated `app.py` file.
4. Follow the provided **GitHub and Streamlit deployment guide** to upload your files to GitHub and deploy the application.
5. Test the deployed Streamlit application and verify that the key workflows are working as expected.

### app.py

In [ ]:
# paste the app.py code and comment it out

### requirements.txt

In [ ]:
# paste the requirement.txt code and comment it out

### Test Case Screenshot

- Add the screenshots of the deployed app executed for all 5 test cases.

## **Actionable Insights and Business Recommendations**

### Actionable Insights


-
-

### Recommendations

-
-